In [2]:
pip install matplotlib-venn

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
  Created wheel for matplotlib-venn: filename=matplotlib_venn-1.1.2-py3-none-any.whl size=45388 sha256=f2b80d6cdf80a486654aa5985cb0192d25fe3dba23b3890360ae0dd525d343ed
  Stored in directory: /home/jeff/.cache/pip/wheels/c2/47/0c/f014c55a1cfd56dce41a1cafd23e3c590652b5e71330cc181c
Successfully built matplotlib-venn
Note: you may need to restart the kernel to use updated packages.


In [1]:
"""
compare_models.py

Compares two BioDiscoveryAgent runs (e.g. two different models) by loading
their sampled_genes_N.npy output files, scoring each against the dataset's
ground-truth hit list, and producing comparison plots.

Usage (run from the BioDiscoveryAgent repo root, so relative paths resolve):

    python compare_models.py \
        --pred_a sonnet_IFNG/test/sampled_genes_5.npy --label_a Sonnet \
        --pred_b haiku_IFNG_rerun/test/sampled_genes_5.npy  --label_b Haiku \
        --dataset IFNG

Outputs:
    hit_ratio_comparison.png   - grouped bar chart of All-genes vs Non-essential
                                  hit ratio for each model
    overlap_breakdown.png      - stacked bar showing shared vs model-only hit
                                  contributions
    gene_overlap_venn.png      - simple two-circle Venn diagram of the two
                                  predicted gene sets
"""

import argparse
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib_venn import venn2  # optional; falls back gracefully if missing


def load_gene_set(path):
    arr = np.load(path, allow_pickle=True)
    return set(arr.tolist())


def compute_hit_ratios(pred_set, topmovers_set, essential_set):
    hits_all = pred_set & topmovers_set
    pred_ne = pred_set - essential_set
    topmovers_ne = topmovers_set - essential_set
    hits_ne = pred_ne & topmovers_ne
    return {
        "hits_all": len(hits_all),
        "total_all": len(topmovers_set),
        "ratio_all": len(hits_all) / len(topmovers_set),
        "hits_ne": len(hits_ne),
        "total_ne": len(topmovers_ne),
        "ratio_ne": len(hits_ne) / len(topmovers_ne),
    }


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("--pred_a", required=True, help="Path to first model's sampled_genes_*.npy")
    parser.add_argument("--pred_b", required=True, help="Path to second model's sampled_genes_*.npy")
    parser.add_argument("--label_a", default="Model A", help="Display name for the first model")
    parser.add_argument("--label_b", default="Model B", help="Display name for the second model")
    parser.add_argument("--dataset", default="IFNG", help="Dataset name (matches datasets/ground_truth_<name>.csv)")
    parser.add_argument("--essential_file", default="CEGv2.txt", help="Path to the essential-genes list")
    parser.add_argument("--outdir", default=".", help="Directory to save output plots")
    args = parser.parse_args()

    # --- Load predictions and ground truth ---
    pred_a = load_gene_set(args.pred_a)
    pred_b = load_gene_set(args.pred_b)

    topmovers = set(np.load(f"datasets/topmovers_{args.dataset}.npy", allow_pickle=True).tolist())
    essential = set(pd.read_csv(args.essential_file, delimiter="\t")["GENE"].tolist())

    stats_a = compute_hit_ratios(pred_a, topmovers, essential)
    stats_b = compute_hit_ratios(pred_b, topmovers, essential)

    print(f"{args.label_a}: All hits = {stats_a['hits_all']}/{stats_a['total_all']} "
          f"({stats_a['ratio_all']:.4f})  |  N/E hits = {stats_a['hits_ne']}/{stats_a['total_ne']} "
          f"({stats_a['ratio_ne']:.4f})")
    print(f"{args.label_b}: All hits = {stats_b['hits_all']}/{stats_b['total_all']} "
          f"({stats_b['ratio_all']:.4f})  |  N/E hits = {stats_b['hits_ne']}/{stats_b['total_ne']} "
          f"({stats_b['ratio_ne']:.4f})")

    # =========================================================
    # Plot 1: Grouped bar chart of hit ratios (All vs Non-essential)
    # =========================================================
    fig, ax = plt.subplots(figsize=(6, 5))
    categories = ["All genes", "Non-essential"]
    a_vals = [stats_a["ratio_all"], stats_a["ratio_ne"]]
    b_vals = [stats_b["ratio_all"], stats_b["ratio_ne"]]

    x = np.arange(len(categories))
    width = 0.35
    ax.bar(x - width / 2, a_vals, width, label=args.label_a)
    ax.bar(x + width / 2, b_vals, width, label=args.label_b)

    y_pad = max(a_vals + b_vals) * 0.02
    for i, v in enumerate(a_vals):
        ax.text(x[i] - width / 2, v + y_pad, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    for i, v in enumerate(b_vals):
        ax.text(x[i] + width / 2, v + y_pad, f"{v:.3f}", ha="center", va="bottom", fontsize=9)
    ax.set_ylim(0, max(a_vals + b_vals) * 1.15)

    ax.set_xticks(x)
    ax.set_xticklabels(categories)
    ax.set_ylabel("Hit ratio")
    ax.set_title(f"Hit ratio comparison — {args.dataset}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/hit_ratio_comparison.png", dpi=150)
    plt.close(fig)

    # =========================================================
    # Plot 2: Breakdown of hits into shared vs. model-only genes
    # =========================================================
    shared_hits = (pred_a & pred_b) & topmovers
    a_only_hits = (pred_a - pred_b) & topmovers
    b_only_hits = (pred_b - pred_a) & topmovers

    fig, ax = plt.subplots(figsize=(6, 5))
    labels = [args.label_a, args.label_b]
    shared_vals = [len(shared_hits), len(shared_hits)]
    unique_vals = [len(a_only_hits), len(b_only_hits)]

    ax.bar(labels, shared_vals, label="Shared hit genes", color="#4C72B0")
    ax.bar(labels, unique_vals, bottom=shared_vals, label="Model-only hit genes", color="#DD8452")

    for i, (s, u) in enumerate(zip(shared_vals, unique_vals)):
        ax.text(i, s / 2, str(s), ha="center", va="center", color="white", fontsize=10)
        ax.text(i, s + u / 2, str(u), ha="center", va="center", color="white", fontsize=10)

    ax.set_ylabel("Number of true-hit genes")
    ax.set_title(f"Hit composition: shared vs. model-specific — {args.dataset}")
    ax.legend()
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/overlap_breakdown.png", dpi=150)
    plt.close(fig)

    # =========================================================
    # Plot 3: Venn diagram of the two predicted gene sets
    # =========================================================
    fig, ax = plt.subplots(figsize=(6, 6))
    try:
        venn2([pred_a, pred_b], set_labels=(args.label_a, args.label_b), ax=ax)
        ax.set_title(f"Predicted gene set overlap — {args.dataset}")
    except NameError:
        # matplotlib_venn not installed; fall back to a text summary plot
        ax.axis("off")
        overlap = len(pred_a & pred_b)
        union = len(pred_a | pred_b)
        ax.text(0.5, 0.6, f"{args.label_a} only: {len(pred_a - pred_b)}", ha="center", fontsize=12)
        ax.text(0.5, 0.5, f"Shared: {overlap}", ha="center", fontsize=12)
        ax.text(0.5, 0.4, f"{args.label_b} only: {len(pred_b - pred_a)}", ha="center", fontsize=12)
        ax.text(0.5, 0.25, "(install matplotlib-venn for a real Venn diagram:\npip install matplotlib-venn)",
                ha="center", fontsize=9, color="gray")
        ax.set_title(f"Predicted gene set overlap — {args.dataset}")
    fig.tight_layout()
    fig.savefig(f"{args.outdir}/gene_overlap_venn.png", dpi=150)
    plt.close(fig)

    print("\nSaved plots: hit_ratio_comparison.png, overlap_breakdown.png, gene_overlap_venn.png")


if __name__ == "__main__":
    main()

usage: ipykernel_launcher.py [-h] --pred_a PRED_A --pred_b PRED_B
                             [--label_a LABEL_A] [--label_b LABEL_B]
                             [--dataset DATASET]
                             [--essential_file ESSENTIAL_FILE]
                             [--outdir OUTDIR]
ipykernel_launcher.py: error: the following arguments are required: --pred_a, --pred_b


SystemExit: 2

/home/jeff/miniconda3/lib/python3.12/site-packages/IPython/core/interactiveshell.py:3585: UserWarning: To exit: use 'exit', 'quit', or Ctrl-D.
  warn("To exit: use 'exit', 'quit', or Ctrl-D.", stacklevel=1)
